In [1]:
import pandas as pd
import numpy as np
# from nltk.tokenize import word_tokenize
# from nltk.stem import WordNetLemmatizer
# import nltk

In [2]:
# Load the datasets
train = pd.read_csv('./data/train_debiased_1000.csv')
test = pd.read_csv('./data/test_debiased_1000.csv')
val = pd.read_csv('./data/val_debiased_1000.csv')

# Split datasets into X and y
X_train = train['debiased_post'].fillna("")
y_train = train['label']

X_test = test['debiased_post'].fillna("")
y_test = test['label']

X_val = val['debiased_post'].fillna("")
y_val = val['label']

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorize text
tfidf = TfidfVectorizer(max_features=1000)
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()
X_val_tfidf = tfidf.transform(X_val).toarray()


Hypertuning 

In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

# Initialize the model
xgb_clf = XGBClassifier(
    objective='multi:softmax',
    num_class=3,
    eta= 0.3,
    eval_metric = 'mlogloss',
    seed=42
)

# Define the parameter grid
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0, 1, 5]
}

# Perform the random search
grid_search = GridSearchCV(
    estimator=xgb_clf,
    param_grid=param_grid,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train_tfidf, y_train)

# Best parameters and score
best_params = grid_search.best_params_
best_score = grid_search.best_score_
print("Best Parameters:", best_params)
print("Best Accuracy:", best_score)

Training the model

In [ ]:
import xgboost as xgb

# Convert data to DMatrix
dtrain = xgb.DMatrix(X_train_tfidf, label=y_train)
dtest = xgb.DMatrix(X_test_tfidf, label=y_test)
dval = xgb.DMatrix(X_val_tfidf, label=y_val)

evals_result = {}

model = xgb.train(best_params, dtrain, num_boost_round=500, evals=[(dval, 'test')], early_stopping_rounds=50, evals_result=evals_result)


Evaluate accuracy

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# Predict on the test set
y_test_pred = model.predict(dtest)
y_test_pred = y_test_pred.astype(int)

# Compute accuracy
accuracy_test = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {accuracy_test * 100:.2f}%")

# Compute F1 Score
f1_test = f1_score(y_test, y_test_pred, average='weighted')  # Use 'weighted' for multi-class
print(f"Test F1 Score: {f1_test:.2f}")


Global feature importance

In [ ]:
# Get feature importance using 'gain'
importance = model.get_score(importance_type='gain')

# Sort the features by importance score in descending order and get top 10
sorted_importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:10]

# Get the feature names from tf-idf
tfidf_feature_names = tfidf.get_feature_names_out()

# Print top 10 features with their importance scores
print("Top 10 Features by Importance:")
for feature_code, score in sorted_importance:
    # Extract the feature index from the code (e.g., f255 -> 255)
    feature_idx = int(feature_code[1:])  # Remove 'f' and convert to integer
    
    feature_name = tfidf_feature_names[feature_idx]

    # # Check if the feature code corresponds to a text feature or numeric feature
    # if feature_idx < len(tfidf_feature_names):  # Text feature
    #     feature_name = tfidf_feature_names[feature_idx]
    # else:  # Numeric feature (based on the position in the combined list)
    #     feature_name = numeric_feature_names[feature_idx - len(tfidf_feature_names)]

    print(f"{feature_name}: {score}")



## Finding biased words using SHAP scores

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
explanation = explainer(dtrain)

shap_values = explanation.values

Plotting shap summary plots for each label

In [ ]:
feature_names = list(tfidf_feature_names)
for label in [0,1,2]:
    print(f"Shap summary plot for Label {label}")
    shap.summary_plot(shap_values[:,:,label],feature_names=feature_names)

Aggregated SHAP summary plot for all 3 labels

In [ ]:
shap_values_aggregated = np.mean(np.abs(shap_values), axis=2)
shap.summary_plot(shap_values_aggregated, feature_names=feature_names)

Aggregate shap scores for each label

In [10]:
n_labels = shap_values.shape[2]

# Step 1: Aggregate SHAP values across samples for each feature and label
shap_aggregated = {}
for label_idx in range(n_labels):
    shap_aggregated[label_idx] = np.mean(np.abs(shap_values[:, :, label_idx]), axis=0)

Print biased words for each label

In [ ]:
# Assuming shap_aggregated is a dictionary: {label_idx: array of n_features}
# and feature_names is a list of feature names

n_labels = len(shap_aggregated)  # Number of labels
n_features = len(feature_names)  # Number of features (words)

# Dictionaries to store biased words for each label
biased_words = {label: [] for label in range(n_labels)}
max_other_zero_words = {label: [] for label in range(n_labels)}

# Define a relative difference threshold 
relative_threshold = 1

for word_idx, word in enumerate(feature_names):
    # Get scores for the current word for all labels
    scores = {label: shap_aggregated[label][word_idx] for label in range(n_labels)}

    for label in scores:
        # Get scores for other labels
        other_labels = [l for l in scores if l != label]
        max_other_score = max([scores[l] for l in other_labels], default=0)

        # # Separate words where max_other_score = 0
        # if max_other_score == 0:
        #     if scores[label] > 0:
        #         max_other_zero_words[label].append((word, scores[label]))
        #     continue

        # Compute the relative difference
        relative_diff = (scores[label] - max_other_score) / max_other_score

        # Check if it exceeds the threshold
        if relative_diff > relative_threshold:
            biased_words[label].append((word, scores[label], max_other_score, relative_diff))

# Sort biased words for each label by relative difference in descending order
for label in biased_words:
    biased_words[label].sort(key=lambda x: x[3], reverse=True)  # Sort by relative_diff (4th element in tuple)

# # Sort words with max_other_score = 0 by label score in descending order
# for label in max_other_zero_words:
#     max_other_zero_words[label].sort(key=lambda x: x[1], reverse=True)  # Sort by label_score (2nd element in tuple)

# Output biased words for each label
for label, words in biased_words.items():
    print(f"\nBiased words for label {label} (with non-zero max_other_score):")
    for word, label_score, max_other_score, relative_diff in words:
        print(f"  {word} - Label Score: {label_score:.6f}, Max Other Score: {max_other_score:.6f}, Relative Difference: {relative_diff:.2%}")

# # Output words with max_other_score = 0 for each label
# for label, words in max_other_zero_words.items():
#     print(f"\nWords with max_other_score = 0 for label {label} (sorted by Label Score):")
#     for word, label_score in words:
#         print(f"  {word} - Label Score: {label_score:.6f}")


In [ ]:
print(len(biased_words[0]))
print(len(biased_words[1]))
print(len(biased_words[2]))

Removing biased words

In [ ]:
# Define thresholds and corresponding filenames
thresholds = [10, 5, 1]
filenames = ['train.csv', 'test.csv', 'val.csv']

# Define a function to remove biased words from a single post
def remove_biased_words(text, biased_words):
    words = text.split()  # Tokenize text into words
    filtered_words = [word for word in words if word not in biased_words]
    return ' '.join(filtered_words)  # Recombine filtered words into a sentence

# Loop over each threshold
for cutoff_threshold in thresholds:
    # Combine biased words based on the relative difference threshold
    biased_word_set = set()
    for label_words in biased_words.values():
        for word, _, _, relative_diff in label_words:
            if relative_diff > cutoff_threshold:
                biased_word_set.add(word)

    for filename in filenames:
        df = pd.read_csv(filename)
        # Apply the function to the 'processed_post' column
        df['debiased_post'] = df['processed_post'].apply(
            lambda x: remove_biased_words(x, biased_word_set)
        )
        name = f'{filename.split('.')[0]}_debiased_{cutoff_threshold}00.csv'
        # Save the cleaned dataset to a file
        df.to_csv(name, index=False)

        print(f"Biased words with relative difference > {cutoff_threshold}00% have been removed. Saved to '{filename.split('.')[0]}_debiased_{cutoff_threshold}00'.csv.")


# Running on debiased dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score

for cutoff_threshold in thresholds:
    train = pd.read_csv(f'./data/train_debiased_{cutoff_threshold}00.csv')
    test = pd.read_csv(f'./data/test{cutoff_threshold}00.csv')
    val = pd.read_csv(f'./data/val_debiased_{cutoff_threshold}00.csv')

    # Split datasets into X and y
    X_train = train['debiased_post']
    y_train = train['label']

    X_test = test['debiased_post']
    y_test = test['label']

    X_val = val['debiased_post']
    y_val = val['label']

    
    # Vectorize text
    tfidf = TfidfVectorizer(max_features=1000)
    X_train_tfidf = tfidf.fit_transform(X_train).toarray()
    X_test_tfidf = tfidf.transform(X_test).toarray()
    X_val_tfidf = tfidf.transform(X_val).toarray()

    # Initialize the model
    xgb_clf = XGBClassifier(
        objective='multi:softmax',
        num_class=3,
        eta= 0.3,
        eval_metric = 'mlogloss',
        seed=42
    )

    # Define the parameter distribution
    param_dist = {
        'max_depth': [4, 6, 8],
        'learning_rate': [0.01, 0.1, 0.3],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0],
        'gamma': [0, 1, 5]
    }

    # Perform the random search
    grid_search = GridSearchCV(
        estimator=xgb_clf,
        param_grid=param_dist,
        scoring='accuracy',
        # verbose=1,
        n_jobs=-1,
    )

    grid_search.fit(X_train_tfidf, y_train)

    # Best parameters and score
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    print("Best Parameters:", best_params)
    print("Best Accuracy:", best_score)


    # Convert data to DMatrix
    dtrain = xgb.DMatrix(X_train_tfidf, label=y_train)
    dtest = xgb.DMatrix(X_test_tfidf, label=y_test)
    dval = xgb.DMatrix(X_val_tfidf, label=y_val)

    model = xgb.train(best_params, dtrain, num_boost_round=500, evals=[(dval, 'test')], early_stopping_rounds=50, evals_result=evals_result)


    # Predict on test  set
    y_test_pred = model.predict(dtest)
    y_test_pred = y_test_pred.astype(int)

    # Compute accuracy
    accuracy_test = accuracy_score(y_test, y_test_pred)
    print(f"Test Accuracy: {accuracy_test * 100:.2f}% for {cutoff_threshold}")

    # Compute F1 Score
    f1_test = f1_score(y_test, y_test_pred, average='weighted')  # Use 'weighted' for multi-class
    print(f"Test F1 Score: {f1_test:.2f}")

In [10]:
# thresholds = [10, 5, 1]
# from sklearn.model_selection import RandomizedSearchCV
# from xgboost import XGBClassifier
# import xgboost as xgb
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics import accuracy_score



# for cutoff_threshold in thresholds:
#     train = pd.read_csv(f'train_debiased_{cutoff_threshold}00.csv')
#     test = pd.read_csv(f'test_debiased_{cutoff_threshold}00.csv')
#     val = pd.read_csv(f'val_debiased_{cutoff_threshold}00.csv')

#     # Split datasets into X and y
#     X_train = train['debiased_post'].fillna('')
#     y_train = train['label']

#     X_test = test['debiased_post'].fillna('')
#     y_test = test['label']

#     X_val = val['debiased_post'].fillna('')
#     y_val = val['label']

    
#     # Vectorize text
#     tfidf = TfidfVectorizer(max_features=1000)
#     X_train_tfidf = tfidf.fit_transform(X_train).toarray()
#     X_test_tfidf = tfidf.transform(X_test).toarray()
#     X_val_tfidf = tfidf.transform(X_val).toarray()

    

#     # Convert data to DMatrix
#     dtrain = xgb.DMatrix(X_train_tfidf, label=y_train)
#     dtest = xgb.DMatrix(X_test_tfidf, label=y_test)
#     dval = xgb.DMatrix(X_val_tfidf, label=y_val)

#     params = {
#     'objective': 'multi:softmax',  # Multi-class classification
#     'num_class': 3,               # Number of classes (center, left, right)
#     'max_depth': 6,
#     'learning_rate': 0.01,
#     'subsample': 0.8,
#     'colsample_bytree': 0.8,
#     'gamma': 0,
#     'eta': 0.3,
#     'eval_metric': 'mlogloss',
#     'seed': 42
#     }

#     evals_result = {}

#     model = xgb.train(params, dtrain, num_boost_round=500, evals=[(dval, 'test')], early_stopping_rounds=50, evals_result=evals_result)


#     # Predict on test  set
#     y_test_pred = model.predict(dtest)

#     # Convert predictions to integer type (as XGBoost predictions are floats)
#     y_test_pred = y_test_pred.astype(int)

#     # Compute accuracy
#     accuracy_test = accuracy_score(y_test, y_test_pred)
#     print(f"Test Accuracy: {accuracy_test * 100:.2f}%  for {cutoff_threshold}")

[0]	test-mlogloss:1.09854
[1]	test-mlogloss:1.09847
[2]	test-mlogloss:1.09833
[3]	test-mlogloss:1.09823
[4]	test-mlogloss:1.09818
[5]	test-mlogloss:1.09809
[6]	test-mlogloss:1.09801
[7]	test-mlogloss:1.09793
[8]	test-mlogloss:1.09787
[9]	test-mlogloss:1.09781
[10]	test-mlogloss:1.09776
[11]	test-mlogloss:1.09767
[12]	test-mlogloss:1.09761
[13]	test-mlogloss:1.09757
[14]	test-mlogloss:1.09754
[15]	test-mlogloss:1.09747
[16]	test-mlogloss:1.09742
[17]	test-mlogloss:1.09733
[18]	test-mlogloss:1.09728
[19]	test-mlogloss:1.09724
[20]	test-mlogloss:1.09721
[21]	test-mlogloss:1.09713
[22]	test-mlogloss:1.09708
[23]	test-mlogloss:1.09705
[24]	test-mlogloss:1.09699
[25]	test-mlogloss:1.09693
[26]	test-mlogloss:1.09687
[27]	test-mlogloss:1.09684
[28]	test-mlogloss:1.09678
[29]	test-mlogloss:1.09671
[30]	test-mlogloss:1.09668
[31]	test-mlogloss:1.09665
[32]	test-mlogloss:1.09666
[33]	test-mlogloss:1.09661
[34]	test-mlogloss:1.09655
[35]	test-mlogloss:1.09652
[36]	test-mlogloss:1.09644
[37]	test-m